# TM1py: Cubes, Dimensions, Hierarchies, Elements

This module is the third chunk of the tm1py course. Readers are assumed
to have absorbed chunks 1 and 2: tm1py is a thin REST wrapper, the
library has Objects and Services, Python objects are snapshots, and
connecting to a server with a `with` block and listing inventory with
`get_all_names()` is routine.

The audience is TM1 expert. The shape of the model, that a cube is a
multidimensional array, that dimensions are axes, that hierarchies
organize the elements along an axis, and that elements are the leaves,
is taken for granted. The point of this chunk is not to teach those
concepts. It is to show how tm1py represents them as Python objects,
and how to navigate from a cube down to an element name without
reaching for MDX or any cell data.

The walk below is short. Get a `Cube`. Read its `dimensions` (a list of
names). Get one of those dimensions. Read its `hierarchies` (a list of
Hierarchy objects). Get one of those hierarchies. Read its element
names. Each step uses one line of code, and each line is the natural
question to ask at that point in the walk.

The walk also makes one design decision tangible: tm1py refers to some
relationships **by name** (the Cube holds dimension names, not Dimension
objects) and others **by embedding** (the Dimension holds Hierarchy
objects directly, not just names). The reason is structural and is
worth a topic of its own.

MDX, cells, writes, and Object construction are deliberately out of
scope here. This is structural metadata only.

The topics below are arranged linearly for review. The same Sales Plan
model from earlier chunks is reused throughout.

---

## Topic list

1. The structural backbone
2. Getting a cube
3. The Cube object and its dimensions attribute
4. Getting a dimension
5. The Dimension object and its hierarchies attribute
6. Getting a hierarchy
7. Element names from a hierarchy
8. By-name vs by-embedding
9. The full walk
10. Real-world design principles
11. Common mistakes

---

## 1. The structural backbone

Cubes, dimensions, hierarchies, and elements are the load bearing
structure of any TM1 model. Every cell is identified by an N-tuple of
elements, one per dimension. Every report, every view, every TI process
ultimately addresses cells through that structure. Anything tm1py reads
or writes also goes through it. So before reading a cell, before
running a process, before doing anything with data, the question is:
what is the shape of this model?

The chunk answers that question with a four step walk down the
structure. From a cube, find its dimension names. From a dimension,
find its hierarchies. From a hierarchy, find its elements. Each step
narrows the focus by one level. None of the steps needs MDX, none
returns a cell value, and none modifies anything on the server.

The same walk is the way most ad hoc tm1py scripts begin. "What
dimensions does this cube have?" "What's the default hierarchy of this
dimension?" "How many elements are in that hierarchy?" Once these
questions can be answered in three lines of code, the rest of the
library begins to feel small.

The running example is the Sales Plan model from previous chunks.
Cubes: `Sales Plan`, `General Ledger`, `HR Plan`. The dimensions of
`Sales Plan`: `Period`, `Region`, `Product`, `Version`, `Measure`. Each
of those dimensions has at least one hierarchy of the same name.

## 2. Getting a cube

The starting point is `tm1.cubes`, the CubeService introduced in chunk
2. Listing all cube names is `tm1.cubes.get_all_names()`. Fetching one
cube by name is `tm1.cubes.get(name)`.

In [ ]:
with TM1Service(**creds) as tm1:
    cube = tm1.cubes.get("Sales Plan")
    print(type(cube))
    # <class 'TM1py.Objects.Cube.Cube'>
    print(cube.name)
    # 'Sales Plan'

The return value is a `Cube` Object: passive, locally held, with
attributes that describe the cube as it stood at the moment of the
fetch. From the chunk 1 mental model, this is a snapshot. The Object
is no longer connected to the server after the call returns.

A `Cube` carries a small, fixed set of attributes:

- `cube.name` is the cube's name as a string.
- `cube.dimensions` is the list of dimensions on the cube, as names.
- `cube.has_rules` is a boolean.
- `cube.rules` is the rules text, present when `has_rules` is True.

That is the whole interface for an inventory style read. The cells of
the cube are not on the Object; cells live behind `tm1.cells` and have
their own service. The shape of a `Cube` Object is metadata only.

The fetch is one HTTP round trip. Each subsequent attribute access on
the returned Object is a local attribute lookup, not a server call.
Reading `cube.dimensions` ten times in a script is essentially free
once the Object exists; what cost a round trip was the fetch itself.

## 3. The Cube object and its dimensions attribute

The most useful attribute on a `Cube` for navigation is `dimensions`.
It is a list of strings: the names of the dimensions on the cube, in
the order they appear in cell tuples.

In [ ]:
with TM1Service(**creds) as tm1:
    cube = tm1.cubes.get("Sales Plan")
    print(cube.dimensions)
    # ['Period', 'Region', 'Product', 'Version', 'Measure']

The order is important. A cell in `Sales Plan` is addressed by a
five tuple in the order Period, Region, Product, Version, Measure. The
order on the cube is fixed at cube creation and changes only if the
cube is rebuilt. Code that addresses cells (chunks 4 and 5) will rely
on this ordering.

The attribute is a list of names, deliberately. It is not a list of
`Dimension` objects. To inspect any dimension in detail, the next step
is to fetch that dimension by name from the DimensionService. This
"by name" relationship is the first half of a design decision worth
making explicit; topic 8 covers the other half and the reason for the
asymmetry.

For inventory questions, the names alone are usually enough.

In [ ]:
with TM1Service(**creds) as tm1:
    cube = tm1.cubes.get("Sales Plan")

    print(f"{cube.name} has {len(cube.dimensions)} dimensions")
    # 'Sales Plan' has 5 dimensions

    if "Currency" in cube.dimensions:
        print("currency is one of the cube's axes")

A note on cost: `tm1.cubes.get(name)` fetches the cube's metadata,
which is small. It does not fetch any of the dimensions referenced by
the cube. Walking from the cube into a particular dimension's contents
is an explicit further fetch, paid for one round trip at a time.

## 4. Getting a dimension

The next step in the walk uses `tm1.dimensions`, the DimensionService.
The shape mirrors `tm1.cubes` exactly: `get_all_names()` for the
inventory, `get(name)` for one Object.

In [ ]:
with TM1Service(**creds) as tm1:
    dim = tm1.dimensions.get("Region")
    print(type(dim))
    # <class 'TM1py.Objects.Dimension.Dimension'>
    print(dim.name)
    # 'Region'

A `Dimension` is, like `Cube`, a passive Object. Its attributes are:

- `dim.name` is the dimension's name.
- `dim.hierarchies` is the list of hierarchies in the dimension, as
  Hierarchy objects (not names).
- `dim.default_hierarchy` is the hierarchy whose name matches the
  dimension's name, if such a hierarchy exists. For most dimensions
  in most TM1 models this is the only hierarchy.

The `hierarchies` attribute is the second half of the design decision
called out earlier. Where `Cube` referred to its dimensions by name,
`Dimension` refers to its hierarchies by embedding. Topic 8 explains
why this is the right asymmetry.

The fetch cost is one round trip and a payload that includes the
dimension's hierarchies, often with elements inlined. The cost depends
on the size of the dimension; a small dimension like `Version` (Plan,
Actual, Forecast) is trivial to fetch, while a dimension with hundreds
of thousands of elements may take noticeable time. For just the
element name list, the smaller `tm1.elements.get_element_names(...)`
call (topic 7) is preferable.

## 5. The Dimension object and its hierarchies attribute

A `Dimension` Object holds its hierarchies as a list of `Hierarchy`
instances, not as a list of strings. Iterating the list and reading
each hierarchy's `name` is the canonical way to ask "what hierarchies
does this dimension have?"

In [ ]:
with TM1Service(**creds) as tm1:
    dim = tm1.dimensions.get("Region")
    print([h.name for h in dim.hierarchies])
    # ['Region']

In most TM1 models, a dimension has exactly one hierarchy whose name
matches the dimension's name; this is the default hierarchy, and it is
what TM1 used to expose before alternate hierarchies were added to the
product. Models that use alternate hierarchies (a `Region` dimension
with a `Region`, a `Sales Region`, and a `Service Region` hierarchy)
will see more entries in the list.

The default hierarchy can also be accessed directly, without iterating:

In [ ]:
with TM1Service(**creds) as tm1:
    dim = tm1.dimensions.get("Region")
    default = dim.default_hierarchy
    print(default.name)
    # 'Region'

This is convenient when the script always means the default hierarchy
and does not want to encode the dimension name twice. For models where
the default and alternate hierarchies are both relevant, iterating
`dim.hierarchies` is the right approach.

Each entry in `dim.hierarchies` is a fully formed `Hierarchy` Object,
with the same shape as one returned directly by the HierarchyService
(topic 6). The fetch in topic 4 already paid for the hierarchies; no
further round trip is needed to access them. The next round trip is
only needed if a hierarchy not yet loaded must be fetched explicitly,
or if a fresh snapshot is wanted (per the chunk 1 rule, the embedded
hierarchies are as fresh as the moment the dimension was fetched, no
fresher).

## 6. Getting a hierarchy

A hierarchy can also be fetched directly, by name, without going
through its containing dimension. The service is `tm1.hierarchies`,
and the verb takes two arguments: dimension name and hierarchy name,
in that order.

In [ ]:
with TM1Service(**creds) as tm1:
    hier = tm1.hierarchies.get("Region", "Region")
    print(type(hier))
    # <class 'TM1py.Objects.Hierarchy.Hierarchy'>
    print(hier.name, hier.dimension_name)
    # 'Region' 'Region'

The two argument call mirrors the REST URL: a hierarchy is identified
within a dimension, not globally. Two different dimensions can each
have a hierarchy called `Region`, and the dimension argument is what
disambiguates. For most workflows, the hierarchy name is the same as
the dimension name, and the call reads as `get("Region", "Region")`,
which is awkward looking but correct.

A `Hierarchy` Object carries:

- `hier.name`: the hierarchy's name.
- `hier.dimension_name`: the containing dimension's name.
- `hier.elements`: a dict of element name to `Element` object.
- `hier.edges`: a dict of `(parent, child)` tuples to weights, the
  parent child edges that define consolidations.
- `hier.element_attributes`: the hierarchy's element attributes
  (numeric, string, or alias attribute definitions).

The two ways to obtain a Hierarchy, going through `tm1.dimensions.get`
and reading `dim.hierarchies`, or fetching directly via
`tm1.hierarchies.get`, return Objects of the same shape. The choice
between them is a question of which entry point is more convenient at
the call site, and of cost: fetching the dimension once and reading
all of its hierarchies is one round trip; fetching each hierarchy
individually is one round trip per hierarchy.

## 7. Element names from a hierarchy

The leaves of the structure are elements. A `Hierarchy` Object exposes
them through its `elements` attribute, which is a dict keyed by
element name.

In [ ]:
with TM1Service(**creds) as tm1:
    hier = tm1.hierarchies.get("Region", "Region")
    print(list(hier.elements.keys())[:5])
    # ['Europe', 'Americas', 'Asia', 'Pacific', 'Middle East']

Each value in the dict is an `Element` Object with `name` and
`element_type` (Numeric, String, or Consolidated, matching the TM1
element types). For most navigation tasks the names alone are enough
and the dict is consulted via its keys.

For the common case of "I just need the names of the elements in this
hierarchy," the lighter call is on the ElementService. It returns the
list of names directly, without constructing the full `Element`
objects or fetching the edges and attributes.

In [ ]:
with TM1Service(**creds) as tm1:
    names = tm1.elements.get_element_names("Region", "Region")
    print(names[:5])
    # ['Europe', 'Americas', 'Asia', 'Pacific', 'Middle East']

This is the form to reach for first when the question is "what
elements exist?" It is faster, the payload is smaller, and the
returned list is a plain Python list ready for use as input to a
pandera `isin` check, an MDX query parameter, a validation set, or
anything else that wants element names as strings.

The richer `tm1.elements.get_elements(...)` returns full `Element`
objects (with type and attributes), which is the right call when the
type matters (filtering numeric leaves out of a tree that includes
consolidations, for instance) or when element attributes need to be
read.

For the structural walk, names are sufficient.

## 8. By-name vs by-embedding

The walk so far has surfaced an asymmetry that is worth naming and
explaining, because it recurs all over tm1py.

`Cube.dimensions` is a list of dimension **names**, not a list of
`Dimension` objects. To work with one of those dimensions in detail, a
separate `tm1.dimensions.get(name)` call is needed.

`Dimension.hierarchies` is a list of `Hierarchy` **objects**, fully
embedded in the Dimension's payload. No further fetch is needed to
inspect a hierarchy listed there.

The asymmetry is deliberate, and the reason follows directly from how
TM1 models its objects.

A dimension is independent of any cube. The same `Region` dimension
can appear on `Sales Plan`, on `General Ledger`, on `HR Plan`, and on
any number of other cubes. Each cube refers to it by name. If a cube
held the full `Dimension` object, the same elements would be embedded
redundantly in every cube that used the dimension, and updating the
dimension in one place would leave stale copies in the others. So the
cube refers by name, and the dimension is fetched separately when
detail is needed.

A hierarchy, in contrast, is not independent of its dimension. A
hierarchy belongs to exactly one dimension; the same hierarchy cannot
appear under two different dimensions. So embedding the hierarchies
directly inside the dimension's payload is harmless, costs nothing in
deduplication, and saves the user one round trip when navigating the
common case.

The general principle is: tm1py refers by name when the related
object is independent and shared, and refers by embedding when the
related object is wholly owned by its parent. The same pattern applies
to:

- `Process` parameters: by embedding (a parameter belongs to the
  process).
- `View` columns and rows: by embedding (the slice belongs to the
  view).
- `Subset` element list: by embedding (the subset belongs to the
  hierarchy).
- `Chore` task list: a list of process names plus their parameters,
  by name on the process side and by embedding on the parameter side.

Internalizing this distinction makes the rest of the library
predictable. When a fetch returns an Object that holds another concept
by name, expect to make a second call to inspect that concept. When it
holds another concept by embedding, expect that the data is already
present.

## 9. The full walk

Putting the topics together, the full walk from the cube to the
element names of one of its dimensions is short and self contained.

In [ ]:
from TM1py import TM1Service

with TM1Service(**creds) as tm1:
    # Step 1: the cube
    cube = tm1.cubes.get("Sales Plan")
    print(f"{cube.name}: {cube.dimensions}")
    # Sales Plan: ['Period', 'Region', 'Product', 'Version', 'Measure']

    # Step 2: walk into one of its dimensions
    dim = tm1.dimensions.get("Region")
    hierarchy_names = [h.name for h in dim.hierarchies]
    print(f"{dim.name} has hierarchies: {hierarchy_names}")
    # Region has hierarchies: ['Region']

    # Step 3: walk into one of its hierarchies
    hier = tm1.hierarchies.get("Region", "Region")
    sample = list(hier.elements.keys())[:3]
    print(f"{hier.name} starts with: {sample}")
    # Region starts with: ['Europe', 'Americas', 'Asia']

Three round trips: one for the cube, one for the dimension (which also
brings the hierarchies inline as Objects), and one for the hierarchy
when its full element dict is wanted. For a cheaper element name list,
the third call can be replaced by `tm1.elements.get_element_names`,
which returns just the names.

The walk is the answer to "what is the shape of this cube?" and is a
useful starting point for any script that goes on to read or write
cells, because every cell address is a tuple of element names, one
per dimension, in the cube's dimension order.

## 10. Real-world design principles

**Walk top down when the question is structural.** Cube → Dimension →
Hierarchy → Element is the way the TM1 model is laid out and the way
tm1py mirrors it. Top down navigation is one round trip per level, and
each level naturally answers a question worth asking at that level
("what are this cube's axes?", "what hierarchies does this dimension
have?", "what elements live in this hierarchy?"). Skipping straight
to elements without knowing the dimension order is a common shortcut
that produces brittle code that breaks when the cube changes.

**Cache the names you'll consult repeatedly.** Element name lists are
the most reused piece of metadata in any tm1py script, used for
validation, filtering, MDX construction, and dimension membership
checks. Per the snapshot principle from chunk 1, fetch each list once
at the start of the session and store it in a Python set or list.
Refetching mid script is a network round trip plus an unnecessary
opportunity for stale data to creep in.

**Prefer `get_element_names` over `get_elements` when only names are
needed.** The lighter call is several times smaller in payload and
correspondingly faster. Reach for `get_elements` only when the element
type or attributes matter.

**Treat `cube.dimensions` as authoritative for cell tuple ordering.**
The order of names in `cube.dimensions` is the order in which a cell
tuple must list its element names. Hardcoding the order in Python
duplicates a piece of TM1 metadata in source code, where it goes stale
the moment the cube is rebuilt. Read it from the cube each session and
use it.

**Recognize the by-name vs by-embedding pattern when reading the
library.** When an Object refers to a related concept by name, expect
to make another service call to fetch detail. When it refers by
embedding, expect detail already to be present. The pattern is not
arbitrary; it tracks the underlying TM1 ownership model, and once the
mapping is internalized, surprises in the API are rare.

## 11. Common mistakes

A short collection of errors that come up while learning to navigate
tm1py's structural metadata.

**Treating `cube.dimensions` as a list of `Dimension` objects.** It is
a list of names. Trying to access `Dimension` attributes on the
elements of that list raises `AttributeError`.

In [ ]:
# Wrong: cube.dimensions[0] is a string
for dim in cube.dimensions:
    print(dim.name)             # AttributeError: 'str' object has no attribute 'name'

# Correct: fetch each dimension explicitly when detail is needed
for name in cube.dimensions:
    dim = tm1.dimensions.get(name)
    print(dim.name)

**Treating `dim.hierarchies` as a list of names.** It is a list of
`Hierarchy` objects. Comparing them directly to a string never matches.

In [ ]:
# Wrong: comparing a Hierarchy object to a string
if "Region" in dim.hierarchies:        # always False
    ...

# Correct: compare names explicitly
if "Region" in [h.name for h in dim.hierarchies]:
    ...

**Hardcoding cube dimension order in Python.** The order is on the
cube; copying it into a Python literal duplicates the metadata and
goes stale silently when the cube changes.

In [ ]:
# Wrong: order copied from the cube into source
DIM_ORDER = ["Period", "Region", "Product", "Version", "Measure"]

# Correct: read the order from the cube once
cube = tm1.cubes.get("Sales Plan")
dim_order = cube.dimensions

**Calling `tm1.dimensions.get` just to read element names.** A full
dimension fetch can be heavy if the dimension is large and the
hierarchies bring elements inline. For the names alone, the
ElementService is lighter.

In [ ]:
# Wrong: pulls the whole dimension just for names
dim = tm1.dimensions.get("Region")
names = list(dim.default_hierarchy.elements.keys())

# Correct: lighter call
names = tm1.elements.get_element_names("Region", "Region")

**Refetching the same metadata in a loop.** The element list does not
change between iterations of a Python loop; fetch it once outside.

In [ ]:
# Wrong: one fetch per iteration
for product in products:
    regions = tm1.elements.get_element_names("Region", "Region")
    if product_region(product) not in regions:
        ...

# Correct: fetch once
regions = set(tm1.elements.get_element_names("Region", "Region"))
for product in products:
    if product_region(product) not in regions:
        ...

**Forgetting that hierarchy names are case sensitive.** TM1 element,
dimension, and hierarchy names are case sensitive in the REST API,
and therefore in tm1py. `"Region"` and `"region"` are different
hierarchies as far as the server is concerned.

In [ ]:
# Wrong: lowercase name does not match the actual hierarchy
hier = tm1.hierarchies.get("Region", "region")
# raises: Hierarchy 'region' not found in Dimension 'Region'

# Correct: use the canonical casing
hier = tm1.hierarchies.get("Region", "Region")